# Notebook05 staged evaluation and scientific readout

This module exports evaluation/report definitions only. Running it top-to-bottom does not extract features, fit probes, calibrate a model, or run an optimizer. Notebook05's controller invokes the exported functions only after the user runs the study.

Inputs: a registered 1,000-update stage plan; immutable initial EMA teachers and full-state checkpoints at updates 300 and 1,000; an audited dataset bundle with explicit `probe_train_ids` (the original 10,100 labeled images), `dev_ids` (the original 2,020 development images), and separate pretraining `train_ids`. Expanded pretraining never expands labeled probe fitting. The official heldout set remains unused.

Outputs: per-stage CPU feature caches and verified fixed final epoch-20 probes, full development predictions, checkpoint/cache identities, learning gates, seed contrasts, static figures, and Markdown/JSON reports. The combined report isolates within-seed resolution and data-volume effects, including changes in the architecture's initialization advantage. Training-step time excludes evaluation and checkpoint overhead; no unmeasured wall time is invented.

Expected runtime: evaluation of 18 encoders per stage is typically minutes to tens of minutes, depending on input resolution and extraction batch size. The registered controller deadline is checked during each extraction batch and every 50 probe updates. Full probe checkpoints are stored in `checkpoints/<run_id>/stages/<stage_id>/probes/<model_id>/`; latest two and best are retained through audited helpers. Feature extraction saves progress every 32 batches or 15 minutes. One evaluation writer per stage.

Resume exactly: import only tagged `ahr-export` cells and call `evaluate_stage` with the identical root, stage artifact, plan, bundle, completed training summary, and original deadline. A changed identity or corrupt committed cache/checkpoint raises. The controller owns the hard deadline and detached tmux session.

The provisional learning gate requires the semantic model's final-minus-own-initialization paired three-seed 95% t interval to have a positive lower bound, and finite representations that are not exactly constant across development images. Effective rank is descriptive only. This exploratory gate is not a population-level theorem; three-seed intervals are fragile and development membership has been reused.


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat
ROOT = Path(os.environ.get("FMM_JEPA_ROOT", "/home/benjamin/workspace/ahr_go/fmm_jeppa")).resolve()
_boot = {"__file__": str(ROOT / "fmm_jepa/bootstrap.ipynb")}
for _cell in nbformat.read(ROOT / "fmm_jepa/bootstrap.ipynb", as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, "bootstrap#ahr-export", "exec"), _boot)
_boot["install_notebook_package"](ROOT)


In [ ]:
from pathlib import Path
from contextlib import contextmanager
import copy, gc, hashlib, itertools, json, math, time
import numpy as np
import torch
from torch.nn import functional as F
from fmm_jepa.training import atomic_json, digest_file, definition_hashes, run_writer, write_csv
from fmm_jepa.semantic_pilot import build_model, configure_environment
from fmm_jepa.semantic_evaluation import _DeadlineDataset, _deadline_check, _release_semantic
from fmm_jepa.food101_data import EvaluationDataset, preprocessing_descriptor
from fmm_jepa.food101_evaluation import (encoder_state_hash, stable_digest, extract_features, validate_feature_cache,
    fit_linear_probe, _load_probe_checkpoint, fit_standardization, standardize, classification_metrics,
    prediction_rows, atomic_torch, embedding_diagnostics)

STAGE_ROLES = ("learning", "resolution", "data_volume")
KINDS = ("vit", "semantic")
MILESTONES = (300, 1000)
RESUME_NOTEBOOK = "notebooks/05_Learning_and_Resolution_Scaling.ipynb"
RESUME_CELL = "launch-staged-study"

def _canonical(value):
    # Normalize object-key types before hashing to avoid integer/string key ordering drift.
    return json.loads(json.dumps(value))

def _digest(value):
    return stable_digest(_canonical(value))

def _read(path):
    return json.loads(Path(path).read_text())

def _verified_record(record):
    path = Path(record["path"])
    if digest_file(path) != record["sha256"]:
        raise ValueError("Evidence file checksum changed: " + str(path))
    return path

def _state_record(path):
    return {"path": str(Path(path).resolve()), "sha256": digest_file(path)}

def _evaluation_export_hash(root):
    doc = _read(Path(root) / "experiments/05_scaling/evaluation.ipynb")
    return _digest(["".join(c["source"]) for c in doc["cells"]
        if c["cell_type"] == "code" and "ahr-export" in c.get("metadata", {}).get("tags", [])])

def _seed_statistics(rows, value_key="delta_percentage_points"):
    if len(rows) != 3 or len({int(row["seed"]) for row in rows}) != 3:
        raise ValueError("This registered study requires exactly three independent paired training seeds")
    values = [float(row[value_key]) for row in rows]
    if not all(math.isfinite(value) for value in values): raise ValueError("Nonfinite seed effect")
    mean = sum(values) / 3
    sd = math.sqrt(sum((value - mean)**2 for value in values) / 2)
    # Exact inverse CDF for t with 2 degrees of freedom: F(t)=1/2+t/(2*sqrt(t*t+2)).
    quantile = math.sqrt(2 * .95**2 / (1 - .95**2))
    half = quantile * sd / math.sqrt(3)
    return {"n_training_seeds": 3, "mean_percentage_points": mean, "sd_percentage_points": sd,
        "approximate_t_ci95_percentage_points": [mean-half, mean+half],
        "range_percentage_points": [min(values), max(values)],
        "positive_seeds": sum(value > 0 for value in values),
        "paired_seed_deltas": [{"seed": int(row["seed"]), "delta_percentage_points": float(row[value_key])} for row in rows],
        "uncertainty": "Paired training seeds are the replication units. Three-seed t intervals are fragile, conditional on this fixed development set, exploratory and not multiplicity-adjusted."}

def _score(saved):
    predictions = saved["probe"]["predictions"]
    return 100 * sum(int(row["correct_top1"]) for row in predictions) / len(predictions)

def _paired_delta(first, second):
    a = {p["sample_id"]: p for p in first["probe"]["predictions"]}
    b = {p["sample_id"]: p for p in second["probe"]["predictions"]}
    if len(a) != len(first["probe"]["predictions"]) or set(a) != set(b) or not a:
        raise ValueError("Paired contrast requires identical unique development IDs")
    if any(a[key]["label"] != b[key]["label"] for key in a):
        raise ValueError("Paired contrast labels differ")
    return 100 * sum(int(a[key]["correct_top1"]) - int(b[key]["correct_top1"]) for key in a) / len(a)

def _fixed_membership(root, bundle):
    if "probe_train_ids" not in bundle:
        raise ValueError("Explicit fixed probe_train_ids are required; never use expanded pretraining train_ids for labels")
    split = _read(Path(root) / "artifacts/notebook_02/food101-pilot-001/split_manifest.json")["selected_ids"]
    if list(bundle["probe_train_ids"]) != split["training"] or list(bundle["dev_ids"]) != split["development"]:
        raise ValueError("Labeled probe or development membership differs from the original audited split")
    if len(split["training"]) != 10100 or len(split["development"]) != 2020:
        raise ValueError("Fixed labeled sample counts changed")
    pretrain, dev, heldout = set(bundle["train_ids"]), set(bundle["dev_ids"]), set(split["heldout"])
    if len(pretrain) != len(bundle["train_ids"]):
        raise ValueError("Duplicate pretraining sample IDs")
    if pretrain & dev or pretrain & heldout or dev & heldout:
        raise ValueError("Pretraining/development/heldout overlap")
    if not set(bundle["probe_train_ids"]).issubset(pretrain):
        raise ValueError("The fixed labeled probe subset must remain inside the pretraining pool")
    return {"probe_train_ids": list(bundle["probe_train_ids"]), "development_ids": list(bundle["dev_ids"]),
        "probe_train_count": 10100, "development_count": 2020, "pretraining_count": len(pretrain),
        "heldout_status": "NOT_RUN", "labels_in_self_supervised_loss": False}

def _stage_jobs(plan, training):
    if int(plan["updates"]) != 1000 or list(plan["milestones"]) != [300,1000]:
        raise ValueError("Registered stage schedule must be 1000 updates with milestones300/1000")
    arms = plan["arms"]
    if len(arms) != 6 or len({a["seed"] for a in arms}) != 3:
        raise ValueError("Each stage requires six arms: three paired seeds x two architectures")
    if training.get("status") not in ("COMPLETE", "COMPLETED"):
        raise ValueError("Only a completed equal-budget stage may enter evaluation")
    if set(training["arms"]) != {arm["id"] for arm in arms}: raise ValueError("Training arms differ from plan")
    jobs = []
    for seed in sorted({a["seed"] for a in arms}):
        if {a["kind"] for a in arms if a["seed"] == seed} != set(KINDS): raise ValueError("Missing paired architecture")
    for arm in arms:
        if arm["config"]["training"]["updates"] != 1000 or arm["config"]["training"]["ema_start"] != .99:
            raise ValueError("Unexpected schedule or EMA factor")
        if arm["config"]["probe"]["epochs"] != 20 or arm["config"]["evaluation"].get("heldout_enabled", False):
            raise ValueError("Probe budget or heldout policy differs")
        record = training["arms"][arm["id"]]
        jobs.append({"model_id": arm["id"] + "-initial", "arm": arm, "updates": 0,
                     "control": "initialization", "state": record["initial_teacher"], "training_cost": None})
        for milestone in MILESTONES:
            state = record["milestones"][str(milestone)]
            if int(state["successful_updates"]) != milestone:
                raise ValueError("Milestone record has the wrong update")
            jobs.append({"model_id": arm["id"] + "-u" + str(milestone), "arm": arm, "updates": milestone,
                         "control": "trained", "state": state, "training_cost": state})
    return jobs


In [ ]:
@contextmanager
def _probe_checkpoint_metadata(plan):
    """Correct resume metadata on every inherited probe checkpoint write."""
    import fmm_jepa.food101_evaluation as module
    original = module._save_probe_checkpoint
    def save(folder, payload, is_best=False):
        result = original(folder, payload, is_best=is_best)
        path = Path(folder) / "probe_manifest.json"
        manifest = _read(path)
        manifest["resume_notebook"] = plan.get("resume_notebook", RESUME_NOTEBOOK)
        manifest["resume_cell"] = plan.get("resume_cell", RESUME_CELL)
        manifest["resume_instructions"] = "Run All with the identical study run ID and registered stage plan; original deadline never resets."
        atomic_json(path, manifest)
        return result
    module._save_probe_checkpoint = save
    try: yield
    finally: module._save_probe_checkpoint = original

def _reconstruct_final_probe(selected, train_cache, dev_cache, predictions, summary):
    """Independently verify final classifier/normalization and every prediction on CPU."""
    for cache in (train_cache, dev_cache): validate_feature_cache(cache, cache["identity"])
    if selected["selection"] != "fixed_final_epoch" or selected["selected_epoch"] != 20:
        raise ValueError("Only the final epoch20 probe is permitted")
    if selected["encoder_checkpoint_hash"] != train_cache["identity"]["checkpoint_hash"]:
        raise ValueError("Final probe encoder identity changed")
    if encoder_state_hash(selected["state_dict"]) != selected["probe_hash"]:
        raise ValueError("Final probe classifier hash changed")
    stats = fit_standardization(train_cache["features"], bool(selected["config"]["standardize"]))
    for key in ("mean", "std"):
        torch.testing.assert_close(stats[key], selected["standardization"][key], rtol=1e-6, atol=1e-7)
    if stats["fit_examples"] != 10100: raise ValueError("Probe normalization used extra labeled images")
    x = (dev_cache["features"].float()-selected["standardization"]["mean"])/selected["standardization"]["std"]
    state = selected["state_dict"]
    with torch.inference_mode():
        logits = torch.cat([F.linear(batch, state["weight"], state["bias"]) for batch in x.split(256)])
    if not torch.isfinite(logits).all(): raise FloatingPointError("Nonfinite final probe logits")
    metrics = classification_metrics(logits, dev_cache["labels"])
    top = logits.topk(5, dim=1).indices
    if [row["sample_id"] for row in predictions] != dev_cache["sample_ids"]:
        raise ValueError("Final prediction IDs/order differ")
    for index, row in enumerate(predictions):
        label = int(dev_cache["labels"][index])
        expected = (int(top[index,0]), top[index].tolist(), int(top[index,0] == label), int((top[index] == label).any()))
        actual = (row["top1_prediction"], row["top5_predictions"], row["correct_top1"], row["correct_top5"])
        if actual != expected or row["label"] != label or row["checkpoint_hash"] != selected["encoder_checkpoint_hash"] or row["probe_hash"] != selected["probe_hash"]:
            raise AssertionError("Final prediction does not reconstruct from the saved CPU classifier")
    for key in ("top1", "top5", "loss"):
        if not math.isclose(metrics[key], summary["metrics"][key], rel_tol=1e-6, abs_tol=1e-7):
            raise AssertionError("Reconstructed final probe metric differs: " + key)
    return {"status": "PASS", "predictions_reconstructed": len(predictions), "device": "cpu",
            "train_only_standardization_recomputed": True, "metrics": metrics}

def _fit_fixed_probe(train_cache, dev_cache, checkpoint_root, config, model_id, encoder, deadline, plan):
    if str(config.get("probe", {}).get("device", "cpu")) != "cpu":
        raise ValueError("Registered frozen linear probes run on CPU")
    folder = Path(checkpoint_root) / model_id
    with _probe_checkpoint_metadata(plan):
        while True:
            _deadline_check(deadline, 2)
            path = folder / "probe_manifest.json"
            updates = int(_read(path)["latest"]["successful_updates"]) if path.exists() else 0
            result = fit_linear_probe(train_cache, dev_cache, checkpoint_root, config, model_id,
                                      encoder=encoder, stop_after_updates=updates+50)
            if result.get("status") != "INTERRUPTED": break
    saved = _load_probe_checkpoint(folder, result["selected_probe"]["identity"])
    if saved["completed_epochs"] != 20 or saved["successful_updates"] != 800 or saved["active_order"] is not None or saved["next_offset"] != 0:
        raise AssertionError("Authoritative final probe checkpoint is incomplete")
    classifier = torch.nn.Linear(train_cache["features"].shape[1], 101)
    classifier.load_state_dict(saved["classifier"])
    classifier.eval()
    with torch.inference_mode():
        logits = torch.cat([classifier(part) for part in standardize(dev_cache["features"], saved["standardization"]).split(256)])
    metrics = classification_metrics(logits, dev_cache["labels"])
    final_rows = [row for row in saved["metrics"] if row["split"] == "development" and row["epoch"] == 20]
    if len(final_rows) != 1 or any(abs(metrics[key]-final_rows[0][key]) > 1e-6 for key in ("top1","top5","loss")):
        raise AssertionError("Final classifier metrics disagree with authoritative epoch20")
    probe_hash = encoder_state_hash(classifier)
    predictions = prediction_rows(logits, dev_cache["labels"], dev_cache["sample_ids"], model_id,
        "development", train_cache["identity"]["checkpoint_hash"], probe_hash, result["summary"]["seed"])
    selected = {"state_dict": saved["classifier"], "standardization": saved["standardization"],
        "config": {**result["summary"]["budget"], "selection": "fixed_final_epoch"},
        "selection": "fixed_final_epoch", "selected_epoch": 20, "probe_hash": probe_hash,
        "encoder_checkpoint_hash": train_cache["identity"]["checkpoint_hash"],
        "identity": result["selected_probe"]["identity"], "source_resume_manifest": str(folder/"probe_manifest.json")}
    summary = {**result["summary"], "selected_epoch": 20, "selection": "fixed_final_epoch",
        "probe_hash": probe_hash, "metrics": metrics, "budget": selected["config"],
        "selected_probe_path": str(folder/"final_epoch_probe.pt"),
        "predictions_path": str(folder/"final_epoch_predictions.csv"),
        "historical_best_dev_artifact_used": False}
    reconciliation = _reconstruct_final_probe(selected, train_cache, dev_cache, predictions, summary)
    atomic_torch(folder/"final_epoch_probe.pt", selected)
    write_csv(folder/"final_epoch_predictions.csv", predictions)
    atomic_json(folder/"final_epoch_summary.json", summary)
    return {"summary": summary, "metrics": result["metrics"], "predictions": predictions,
        "reconciliation": reconciliation, "final_probe": _state_record(folder/"final_epoch_probe.pt")}

def _cached_representation_diagnostics(train_cache, dev_cache, diagnostic_ids):
    features = dev_cache["features"]
    if not torch.isfinite(features).all(): raise FloatingPointError("Nonfinite development representation")
    indices = {sample_id:index for index,sample_id in enumerate(dev_cache["sample_ids"])}
    chosen = [indices[sample_id] for sample_id in diagnostic_ids if sample_id in indices]
    if len(chosen) < 2: chosen = list(range(min(202,len(features))))
    descriptive = embedding_diagnostics(features[chosen])
    return {"finite_train_and_development": bool(torch.isfinite(train_cache["features"]).all() and torch.isfinite(features).all()),
        "development_representation_not_exactly_constant": bool((features != features[0:1]).any()),
        "training_representation_not_exactly_constant": bool((train_cache["features"] != train_cache["features"][0:1]).any()),
        "gate_uses_effective_rank_threshold": False,
        "descriptive": descriptive,
        "gate_interpretation": "Finite and not exactly constant is a bounded numerical sanity check; useful learning is evaluated separately. Effective rank remains advisory."}


In [ ]:
def _verify_plan_sources(root, plan):
    actual = definition_hashes(root)
    for path in sorted(list((Path(root)/"experiments/04_followup").glob("*.ipynb")) + list((Path(root)/"experiments/05_scaling").glob("*.ipynb"))):
        code = ["".join(c["source"]) for c in _read(path)["cells"]
                if c["cell_type"] == "code" and "ahr-export" in c.get("metadata", {}).get("tags", [])]
        actual[str(path.relative_to(root))] = hashlib.sha256("\n".join(code).encode()).hexdigest()
    if plan.get("source_hashes") != actual:
        raise ValueError("Registered Notebook05 source definitions changed")
    if plan.get("consumer_notebook_sha256"):
        path = Path(root)/plan["resume_notebook"]
        code = ["".join(c["source"]) for c in _read(path)["cells"] if c["cell_type"] == "code"]
        if _digest(code) != plan["consumer_notebook_sha256"]:
            raise ValueError("Registered main notebook code changed")
    return actual

def _load_saved_job(summary_path, record, identity_hash, job, membership):
    if digest_file(summary_path) != record["sha256"]:
        raise ValueError("Completed model-evaluation summary checksum differs")
    saved = _read(summary_path)
    if saved["identity_hash"] != identity_hash or saved["model_id"] != job["model_id"]:
        raise ValueError("Completed model-evaluation identity differs")
    caches = {}
    for split, ids in (("train", membership["probe_train_ids"]), ("development", membership["development_ids"])):
        path = _verified_record(saved["feature_caches"][split])
        metadata = _read(path.with_suffix(".json"))
        if metadata["sha256"] != digest_file(path): raise ValueError("Committed feature cache checksum differs")
        cache = torch.load(path, map_location="cpu", weights_only=True)
        validate_feature_cache(cache, metadata["identity"])
        if cache["sample_ids"] != ids: raise ValueError("Reused labeled probe membership differs")
        caches[split] = cache
    selected_path = _verified_record(saved["probe"]["final_probe"])
    selected = torch.load(selected_path, map_location="cpu", weights_only=True)
    _reconstruct_final_probe(selected, caches["train"], caches["development"], saved["probe"]["predictions"], saved["probe"]["summary"])
    return saved

def evaluate_stage(root, artifact_dir, plan, bundle, training, deadline_unix):
    """Evaluate a completed stage; never expand labeled probe IDs or select a dev epoch."""
    root, artifact = Path(root).resolve(), Path(artifact_dir).resolve()
    artifact.relative_to(root)
    artifact.mkdir(parents=True, exist_ok=True)
    with run_writer(artifact/"evaluation_writer"):
        return _evaluate_stage_locked(root, artifact, plan, bundle, training, float(deadline_unix))

def _evaluate_stage_locked(root, artifact, plan, bundle, training, deadline):
    sources = _verify_plan_sources(root, plan)
    membership = _fixed_membership(root, bundle)
    jobs = _stage_jobs(plan, training)
    stage_role = plan.get("stage_role", plan["stage_id"])
    if stage_role not in STAGE_ROLES: raise ValueError("Unknown stage_role")
    if membership["pretraining_count"] != (65600 if stage_role == "data_volume" else 10100):
        raise ValueError("Pretraining count differs from the registered stage role")
    if int(plan["arms"][0]["config"]["image_size"]) != (256 if stage_role == "resolution" else 128):
        raise ValueError("Stage image resolution differs from declared role")
    identity = {"schema": "notebook05-stage-evaluation/v1", "plan": _canonical(plan),
        "source_hashes": sources, "evaluation_export_hash": _evaluation_export_hash(root),
        "dataset_fingerprint": bundle["dataset"].fingerprint, "membership": membership,
        "jobs": _canonical(jobs), "probe_selection": "fixed_final_epoch20", "official_test": "NOT_RUN"}
    identity_hash = _digest(identity)
    manifest_path = artifact/"stage_evaluation_manifest.json"
    if manifest_path.exists():
        manifest = _read(manifest_path)
        if manifest["identity_hash"] != identity_hash: raise ValueError("Stage evaluation identity changed; refusing stale reuse")
    else:
        manifest = {"identity_hash": identity_hash, "identity": identity, "completed": {}, "started_unix": time.time()}
        atomic_json(manifest_path, manifest)
    evaluations = {}
    for job in jobs:
        model_id, arm = job["model_id"], job["arm"]
        config = copy.deepcopy(arm["config"])
        summary_path = artifact/"evaluation"/(model_id+".json")
        _verified_record(job["state"])
        if model_id in manifest["completed"]:
            evaluations[model_id] = _load_saved_job(summary_path, manifest["completed"][model_id], identity_hash, job, membership)
            continue
        _deadline_check(deadline, 5)
        configure_environment(config)
        atomic_json(artifact/"evaluation_status.json", {"status":"RUNNING", "model_id":model_id,
            "completed_models":len(evaluations), "total_models":len(jobs), "updated_unix":time.time()})
        model = encoder = None
        try:
            _release_semantic(config["device"])
            model = build_model(config, arm["kind"]).cpu()
            state = torch.load(job["state"]["path"], map_location="cpu", weights_only=job["updates"] == 0)
            if job["updates"] == 0:
                model.target.load_state_dict(state)
            else:
                if int(state["state"]["successful_updates"]) != job["updates"] or state["config"] != arm["config"]:
                    raise ValueError("Checkpoint milestone/config differs from the registered 1000-update schedule")
                model.load_state_dict(state["model"])
            del state
            encoder = model.target
            model = None
            common = {"dataset_revision":bundle["dataset"].fingerprint, "checkpoint_hash":encoder_state_hash(encoder),
                "encoder_kind":arm["kind"], "preprocessing":preprocessing_descriptor(config),
                "pooling":"mean_final_layernorm_patches",
                "normalization":"encoder final affine LayerNorm then patch mean; train-only per-feature probe standardization"}
            caches, extraction = {}, []
            for split, ids in (("train", membership["probe_train_ids"]), ("development", membership["development_ids"])):
                _deadline_check(deadline, 2)
                dataset = _DeadlineDataset(EvaluationDataset(bundle["dataset"], ids, config), deadline)
                print(stage_role, model_id, split, len(ids), "fixed labeled feature IDs", flush=True)
                cache = extract_features(encoder, dataset, artifact/"features"/model_id,
                    {**common,"sample_ids":ids}, batch_size=int(config["evaluation"]["batch_size"]), device=config["device"])
                caches[split] = cache
                extraction.append({"split":split, "cache_reused":cache["reused"], **cache["performance"]})
            encoder.cpu()
            _release_semantic(config["device"])
            diagnostics = _cached_representation_diagnostics(caches["train"], caches["development"], bundle.get("diagnostic_ids", []))
            result = _fit_fixed_probe(caches["train"], caches["development"],
                root/"checkpoints"/plan["run_id"]/"stages"/plan["stage_id"]/"probes", config, model_id, encoder, deadline, plan)
            saved = {"identity_hash":identity_hash, "model_id":model_id, "arm_id":arm["id"], "kind":arm["kind"],
                "seed":int(arm["seed"]), "stage_role":stage_role, "updates":job["updates"], "control":job["control"],
                "source_encoder":job["state"], "feature_caches":{name:_state_record(cache["path"]) for name,cache in caches.items()},
                "probe":result, "representation_diagnostics":diagnostics, "feature_extraction":extraction,
                "training_cost":job["training_cost"], "heldout_status":"NOT_RUN"}
            write_csv(artifact/"predictions"/(model_id+".csv"), result["predictions"])
            atomic_json(summary_path, saved)
            manifest["completed"][model_id] = _state_record(summary_path)
            atomic_json(manifest_path, manifest)
            evaluations[model_id] = saved
            print(stage_role, model_id, "fixed epoch20 top1", 100*result["summary"]["metrics"]["top1"], flush=True)
        finally:
            model = encoder = None
            _release_semantic(config["device"])
    _deadline_check(deadline)
    if len(evaluations) != 18: raise AssertionError("Stage must have six initial and twelve trained probes")
    probe_inits = {x["probe"]["summary"]["initialization_hash"] for x in evaluations.values()}
    probe_budgets = {_digest(x["probe"]["summary"]["budget"]) for x in evaluations.values()}
    probe_orders = {_digest([(r["epoch"],r["split"],r["permutation_hash"]) for r in x["probe"]["metrics"]]) for x in evaluations.values()}
    if len(probe_inits) != 1 or len(probe_budgets) != 1 or len(probe_orders) != 1:
        raise AssertionError("Probe initialization/budget/order is not identical across stage comparisons")
    scientific = analyze_stage_evidence(evaluations)
    result = {"schema":"notebook05-stage-evaluation/v1", "status":"COMPLETE", "run_id":plan["run_id"],
        "stage_id":plan["stage_id"], "stage_role":stage_role, "identity_hash":identity_hash,
        "plan_hash":_digest(plan), "dataset_fingerprint":bundle["dataset"].fingerprint,
        "membership":membership, "evaluations":evaluations, "milestones":[0,300,1000],
        "schedule_note":"Both milestones belong to the same predeclared 1000-update learning-rate and EMA schedule; no resumed extension of a finished300-step schedule.",
        "image_size":int(plan["arms"][0]["config"]["image_size"]), "pretraining_images":len(bundle["train_ids"]),
        "training_design":{"updates":1000,"effective_batch":plan["arms"][0]["config"]["training"]["microbatch"]*plan["arms"][0]["config"]["training"]["accumulation_steps"],
            "lr":plan["arms"][0]["config"]["training"]["lr"],"ema_start":.99,
            "weight_decay":plan["arms"][0]["config"]["training"]["weight_decay"],
            "warmup_fraction":plan["arms"][0]["config"]["training"]["warmup_fraction"],
            "patch_size":plan["arms"][0]["config"]["model"]["patch_size"],
            "model":copy.deepcopy(plan["arms"][0]["config"]["model"]), "dtype":plan["arms"][0]["config"].get("dtype","float32") },
        "probe_selection":"fixed_final_epoch20", "labels_in_self_supervised_loss":False,
        "official_test":"NOT_RUN", "started_unix":manifest["started_unix"], "completed_unix":time.time(),
        "learning_gate":scientific["learning_gate"], "contrasts":scientific["contrasts"], "chart_rows":scientific["chart_rows"],
        "report_scope":"Three-seed exploratory experiment conditional on reused development membership. Positive raw architecture accuracy may include an initialization advantage."}
    atomic_json(artifact/"stage_evaluation.json", result)
    result["presentation"] = write_stage_report(artifact, result)
    atomic_json(artifact/"stage_evaluation.json", result)
    atomic_json(artifact/"evaluation_status.json", {"status":"COMPLETE", "completed_models":18, "learning_gate":result["learning_gate"], "updated_unix":time.time()})
    return result

def analyze_stage_evidence(evaluations):
    index = {(v["kind"],int(v["seed"]),int(v["updates"])):v for v in evaluations.values()}
    seeds = sorted({key[1] for key in index})
    if len(seeds) != 3 or len(index) != 18: raise ValueError("Stage readout needs exactly three paired seeds and18 encoders")
    contrasts, rows = {}, []
    for kind in KINDS:
        for step in MILESTONES:
            name = kind + "_learning_u" + str(step)
            paired = [{"seed":seed,"delta_percentage_points":_paired_delta(index[(kind,seed,step)],index[(kind,seed,0)])} for seed in seeds]
            contrasts[name] = _seed_statistics(paired)
    for step in (0,300,1000):
        paired = [{"seed":seed,"delta_percentage_points":_paired_delta(index[("semantic",seed,step)],index[("vit",seed,step)])} for seed in seeds]
        contrasts["semantic_minus_vit_u"+str(step)] = _seed_statistics(paired)
        if step:
            paired_gain = [{"seed":seed,"delta_percentage_points":
                _paired_delta(index[("semantic",seed,step)],index[("semantic",seed,0)])-
                _paired_delta(index[("vit",seed,step)],index[("vit",seed,0)])} for seed in seeds]
            contrasts["semantic_minus_vit_learning_gain_u"+str(step)] = _seed_statistics(paired_gain)
    for kind in KINDS:
        for seed in seeds:
            initial = _score(index[(kind,seed,0)])
            for step in (0,300,1000):
                saved = index[(kind,seed,step)]
                cost = saved.get("training_cost") or {}
                perf = cost.get("performance") or {}
                peak = perf.get("peak_cuda_allocated_bytes")
                median = (perf.get("complete_jepa_step_seconds") or {}).get("median")
                seconds = cost.get("cumulative_step_seconds") if step else 0.0
                rows.append({"model_id":saved["model_id"],"kind":kind,"seed":seed,"updates":step,
                    "top1_percent":_score(saved),"initial_top1_percent":initial,"gain_percentage_points":_score(saved)-initial,
                    "image_exposures":int(cost.get("image_exposures",0)),
                    "cumulative_training_step_seconds":seconds,
                    "cumulative_compute_seconds":cost.get("cumulative_compute_seconds") if step else 0.0,
                    "cumulative_data_seconds":cost.get("cumulative_data_seconds") if step else 0.0,
                    "median_training_step_seconds":median,
                    "peak_training_GiB":float(peak)/1024**3 if peak is not None else None,
                    "time_scope":"Sum of recorded training step timings including data preparation; excludes checkpoint/probe overhead. Missing measurements remain UNKNOWN."})
    sanity = {str(seed):{"finite":bool(index[("semantic",seed,1000)]["representation_diagnostics"]["finite_train_and_development"]),
        "not_exactly_constant":bool(index[("semantic",seed,1000)]["representation_diagnostics"]["development_representation_not_exactly_constant"] and index[("semantic",seed,1000)]["representation_diagnostics"]["training_representation_not_exactly_constant"])} for seed in seeds}
    lower = contrasts["semantic_learning_u1000"]["approximate_t_ci95_percentage_points"][0]
    passed = lower > 0 and all(x["finite"] and x["not_exactly_constant"] for x in sanity.values())
    reason = ("Semantic final-minus-own-initialization three-seed t interval has positive lower bound and final representations pass finite/nonconstant checks."
        if passed else "Semantic learning over its own initialization is not established by the predeclared gate, or a final finite/nonconstant check failed; this does not prove learning is absent.")
    gate = {"passed":passed,"status":"PASS" if passed else "LEARNING_UNESTABLISHED","reason":reason,
        "contrast":"semantic_learning_u1000","lower_t_bound_percentage_points":lower,"representation_checks":sanity,
        "effective_rank_used_as_gate":False,"preregistered_rule":"lower95% across-three-seed t bound >0 AND finite and not exactly constant final representations",
        "caution":"Exploratory gate with three seeds on reused development data; no universal learning or superiority guarantee."}
    return {"contrasts":contrasts,"chart_rows":rows,"learning_gate":gate}


In [ ]:
def write_stage_report(artifact_dir, result):
    """Render measured per-seed evidence; no placeholder outcomes are generated."""
    import matplotlib
    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt
    artifact = Path(artifact_dir)
    artifact.mkdir(parents=True, exist_ok=True)
    rows = result["chart_rows"]
    if not rows: raise ValueError("Cannot report an unmeasured stage")
    fixture = result.get("evidence_type") == "SYNTHETIC_CPU_REPORT_FIXTURE"
    title = ("SYNTHETIC REPORT FIXTURE — not experimental results" if fixture else
             f"{result['stage_role']} · {result['image_size']}×{result['image_size']} · {result['pretraining_images']:,} pretraining images")
    colors = {"vit":"#2966A0", "semantic":"#A26822"}
    labels = {"vit":"ViT", "semantic":"Semantic"}
    plt.rcParams.update({"font.size":10, "axes.spines.top":False, "axes.spines.right":False, "figure.facecolor":"white"})
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.7), constrained_layout=True)
    for kind in KINDS:
        seeds = sorted({row["seed"] for row in rows if row["kind"] == kind})
        for seed_index, seed in enumerate(seeds):
            group = sorted([row for row in rows if row["kind"] == kind and row["seed"] == seed], key=lambda row:row["updates"])
            for axis, xfield, yfield in ((axes[0],"updates","top1_percent"), (axes[1],"updates","gain_percentage_points"),
                                        (axes[2],"cumulative_training_step_seconds","top1_percent")):
                values = [(row[xfield],row[yfield]) for row in group if row.get(xfield) is not None and row.get(yfield) is not None]
                if values:
                    xs, ys = zip(*values)
                    if xfield == "cumulative_training_step_seconds": xs = [value/60 for value in xs]
                    axis.plot(xs, ys, marker=("o","s","^")[seed_index], color=colors[kind], alpha=.65,
                              label=labels[kind] if seed_index == 0 else None)
    for axis in axes:
        axis.grid(alpha=.18); axis.set_axisbelow(True)
    axes[0].set(title="Fixed epoch20 probe accuracy", xlabel="Pretraining updates", ylabel="Development top-1 (%)")
    axes[0].set_ylim(bottom=0); axes[0].set_xticks([0,300,1000]); axes[0].legend(frameon=False)
    axes[1].set(title="Gain over own initialization", xlabel="Pretraining updates", ylabel="Top-1 gain (percentage points)")
    axes[1].axhline(0,color="#444444",linewidth=1); axes[1].set_xticks([0,300,1000])
    axes[2].set(title="Quality versus measured training cost", xlabel="Cumulative training step minutes", ylabel="Development top-1 (%)")
    axes[2].set_ylim(bottom=0); axes[2].set_xlim(left=0)
    fig.suptitle(title)
    quality_path = artifact/"stage_quality.png"
    fig.savefig(quality_path,dpi=150);plt.close(fig)
    fig, axes = plt.subplots(1,2,figsize=(10,4.4),constrained_layout=True)
    final = [row for row in rows if row["updates"] == 1000]
    for axis,field,heading,unit in ((axes[0],"median_training_step_seconds","Median complete training step","Seconds per update"),
                                   (axes[1],"peak_training_GiB","Peak allocated training memory","GiB")):
        maximum = 0
        for index,kind in enumerate(KINDS):
            values = [row[field] for row in final if row["kind"] == kind and row.get(field) is not None]
            if values:
                axis.bar(index,float(np.mean(values)),color=colors[kind],alpha=.28)
                axis.scatter([index+(i-(len(values)-1)/2)*.09 for i in range(len(values))],values,color=colors[kind],zorder=3)
                maximum=max(maximum,max(values))
            else: axis.text(index,.05,"UNKNOWN",ha="center",transform=axis.get_xaxis_transform())
        axis.set_xticks(range(2),[labels[k] for k in KINDS]);axis.set_ylim(0,maximum*1.2 if maximum else 1)
        axis.set(title=heading,ylabel=unit);axis.grid(axis="y",alpha=.18);axis.set_axisbelow(True)
    fig.suptitle(title)
    compute_path=artifact/"stage_compute.png";fig.savefig(compute_path,dpi=150);plt.close(fig)
    data_path=artifact/"stage_chart_data.json";atomic_json(data_path,rows)
    figure_manifest={"source":str(data_path),"source_sha256":digest_file(data_path),
        "figures":[str(quality_path),str(compute_path)],"visual_inspection":"PENDING",
        "scope":"Each line/point is one paired training seed; connecting lines are visual guides. Time includes recorded training steps, not probe or checkpoint cost. Missing cost stays UNKNOWN.",
        "evidence_type":result.get("evidence_type","MEASURED_EXPERIMENT")}
    atomic_json(artifact/"stage_figure_manifest.json",figure_manifest)
    lines=["# Notebook05 stage: "+result["stage_role"],"",title,"",
        "Three paired training seeds; the 300- and 1,000-update checkpoints belong to one registered 1,000-update schedule.","",
        "Labeled probe training: **10,100 fixed images**. Development: **2,020 fixed images**. Every probe uses final epoch20; official test: **NOT_RUN**.","",
        "| Architecture | Updates | Mean top-1 | Mean gain over own initialization |", "|---|---:|---:|---:|"]
    for kind in KINDS:
        for step in (0,300,1000):
            group=[row for row in rows if row["kind"]==kind and row["updates"]==step]
            lines.append(f"| {labels[kind]} | {step} | {np.mean([r['top1_percent'] for r in group]):.2f}% | {np.mean([r['gain_percentage_points'] for r in group]):+.2f} pp |")
    lines += ["", "| Paired contrast | Individual seed deltas (pp) | Mean (pp) | Approximate95% seed-t interval (pp) |", "|---|---|---:|---|"]
    for name,value in result["contrasts"].items():
        low,high=value["approximate_t_ci95_percentage_points"]
        deltas=", ".join(f"{row['seed']}: {row['delta_percentage_points']:+.2f}" for row in value["paired_seed_deltas"])
        lines.append(f"| {name} | {deltas} | {value['mean_percentage_points']:+.2f} | [{low:+.2f}, {high:+.2f}] |")
    gate=result["learning_gate"]
    lines += ["",f"Learning gate: **{gate['status']}**. {gate['reason']}","",
        "Effective rank is descriptive; no arbitrary rank threshold controls progression. Nonconstant finite features alone do not establish useful learning.","",
        "![Per-seed quality, own-initialization gain and measured cost](stage_quality.png)","",
        "![Measured step time and memory](stage_compute.png)","",
        "Raw semantic-minus-ViT accuracy and difference in learning gains are reported separately because architectures can differ before pretraining. The initialization contrast is explicitly shown.","",
        "Three-seed t intervals are fragile and conditional on reused development data. These multiple comparisons are exploratory. An interval spanning zero is unresolved, not equivalence. Equal update/image-exposure experiments and measured cost curves do not automatically establish equal-compute superiority.","",
        "Native image resolution and pretraining volume are changed in separate later stages. Expanded pretraining never provides additional labeled examples to the probe."]
    if fixture:lines.insert(2,"**SYNTHETIC CPU REPORT FIXTURE — no training or scientific outcome.**")
    path=artifact/"stage_results.md";path.write_text("\n".join(lines)+"\n")
    return {"report_path":str(path),"figure_paths":[str(quality_path),str(compute_path)],"figure_manifest_path":str(artifact/"stage_figure_manifest.json")}

def combine_stages(root, study_artifact, stage_results, plan=None):
    """Combine measured stages; preserve explicit NOT_RUN outcomes and pairing."""
    root, artifact = Path(root).resolve(), Path(study_artifact).resolve()
    artifact.relative_to(root);artifact.mkdir(parents=True,exist_ok=True)
    results={role:stage_results.get(role,{"status":"NOT_RUN","reason":"No result supplied for this stage"}) for role in STAGE_ROLES}
    completed={role:result for role,result in results.items() if result.get("status")=="COMPLETE"}
    for role,result in completed.items():
        if result["stage_role"] != role:raise ValueError("Stage result role mismatch")
        if result["membership"]["probe_train_count"] != 10100 or result["membership"]["development_count"] != 2020:
            raise ValueError("Fixed labeled evaluation budget changed between stages")
    evidence_types={value.get("evidence_type","MEASURED_EXPERIMENT") for value in completed.values()}
    if len(evidence_types)>1: raise ValueError("Never mix synthetic fixtures with measured research results")
    evidence_type=next(iter(evidence_types),"NO_MEASURED_STAGES")
    effects={}
    baseline=completed.get("learning")
    if baseline:
        baseline_index={(v["kind"],v["seed"],v["updates"]):v for v in baseline["evaluations"].values()}
        for role in ("resolution","data_volume"):
            if role not in completed:continue
            result=completed[role]
            for key in ("probe_train_ids","development_ids"):
                if result["membership"][key] != baseline["membership"][key]:raise ValueError("Cross-stage labeled membership differs")
            if result["dataset_fingerprint"] != baseline["dataset_fingerprint"] or result["training_design"] != baseline["training_design"]:
                raise ValueError("Cross-stage data identity or matched training budget differs")
            if role=="resolution" and result["pretraining_images"] != baseline["pretraining_images"]:
                raise ValueError("Resolution comparison also changed pretraining count")
            if role=="data_volume" and result["image_size"] != baseline["image_size"]:
                raise ValueError("Data-volume comparison also changed resolution")
            index={(v["kind"],v["seed"],v["updates"]):v for v in result["evaluations"].values()}
            if set(index) != set(baseline_index):raise ValueError("Cross-stage paired model/seed/milestone set differs")
            seeds=sorted({key[1] for key in index})
            for kind in KINDS:
                for step in (300,1000):
                    raw=[];gain=[];initial=[]
                    for seed in seeds:
                        effect=_paired_delta(index[(kind,seed,step)],baseline_index[(kind,seed,step)])
                        initialization_effect=_paired_delta(index[(kind,seed,0)],baseline_index[(kind,seed,0)])
                        raw.append({"seed":seed,"delta_percentage_points":effect})
                        gain.append({"seed":seed,"delta_percentage_points":effect-initialization_effect})
                        initial.append({"seed":seed,"delta_percentage_points":initialization_effect})
                    prefix=role+"_minus_learning_"+kind+"_u"+str(step)
                    effects[prefix+"_raw_top1"]=_seed_statistics(raw)
                    effects[prefix+"_learning_gain"]=_seed_statistics(gain)
                    if step==1000:effects[role+"_initialization_effect_"+kind]=_seed_statistics(initial)
            for step in (300,1000):
                interactions=[]
                for seed in seeds:
                    semantic_effect=_paired_delta(index[("semantic",seed,step)],baseline_index[("semantic",seed,step)])
                    vit_effect=_paired_delta(index[("vit",seed,step)],baseline_index[("vit",seed,step)])
                    semantic_initial=_paired_delta(index[("semantic",seed,0)],baseline_index[("semantic",seed,0)])
                    vit_initial=_paired_delta(index[("vit",seed,0)],baseline_index[("vit",seed,0)])
                    interactions.append({"seed":seed,"delta_percentage_points":(semantic_effect-semantic_initial)-(vit_effect-vit_initial)})
                effects[role+"_architecture_interaction_in_learning_gain_u"+str(step)]=_seed_statistics(interactions)
    gate_failed=bool(baseline and not baseline["learning_gate"]["passed"])
    requested_roles=list((plan or {}).get("stage_ids", STAGE_ROLES))
    if any(role not in STAGE_ROLES for role in requested_roles):raise ValueError("Unknown requested stage")
    requested_complete=bool(requested_roles) and all(role in completed for role in requested_roles)
    status=("LEARNING_UNESTABLISHED" if gate_failed else "COMPLETE" if requested_complete else "PARTIAL_WITH_EXPLICIT_NOT_RUN")
    result={"schema":"notebook05-combined-evaluation/v1","status":status,"evidence_type":evidence_type,
        "stage_statuses":{role:{"status":value.get("status","UNKNOWN"),"reason":value.get("reason","")} for role,value in results.items()},
        "requested_stage_roles":requested_roles,"completed_stage_roles":list(completed),"learning_gate":baseline.get("learning_gate") if baseline else None,
        "cross_stage_effects":effects,"stage_reports":{role:value.get("presentation",{}) for role,value in completed.items()},
        "fixed_probe_images":10100,"fixed_development_images":2020,"official_test":"NOT_RUN",
        "pairing":"Every stage effect and interaction is formed within each training seed first, preserving paired covariance. Three seeds, not image count, determine uncertainty.",
        "scope":"Separate resolution/data-volume manipulations at the same registered update and exposure budget. Measured quality-time-memory trajectories are descriptive; equal-compute superiority requires a matched-compute endpoint.",
        "source_stage_identity_hashes":{role:value["identity_hash"] for role,value in completed.items()}}
    lines=["# Notebook05 staged study results","", "Study outcome: **"+status+"**.","",
        "| Stage | Status | Explanation |", "|---|---|---|"]
    if evidence_type=="SYNTHETIC_CPU_REPORT_FIXTURE":lines.insert(2,"**SYNTHETIC CPU REPORT FIXTURE — NOT RESEARCH RESULTS.**")
    for role,value in results.items():lines.append("| "+role+" | "+value.get("status","UNKNOWN")+" | "+value.get("reason", "Completed registered stage" if role in completed else "No measured outcome")+" |")
    if gate_failed:lines += ["",baseline["learning_gate"]["reason"],"","Later NOT_RUN stages have no inferred quality or efficiency result."]
    lines += ["","All stages use the same10,100 labeled probe-training images and2,020 development images. Extra pretraining data never expands the supervised evaluation budget.","",
        "| Cross-stage paired effect | Seed deltas (pp) | Mean (pp) | Approximate95% seed-t interval (pp) |","|---|---|---:|---|"]
    for name,value in effects.items():
        lo,hi=value["approximate_t_ci95_percentage_points"]
        deltas=", ".join(f"{row['seed']}: {row['delta_percentage_points']:+.2f}" for row in value["paired_seed_deltas"])
        lines.append(f"| {name} | {deltas} | {value['mean_percentage_points']:+.2f} | [{lo:+.2f}, {hi:+.2f}] |")
    lines += ["",result["pairing"],"",result["scope"],"",
        "Initialization effects are shown separately. Learning-gain differences subtract each stage's own initialization before pairing. Development reuse, three seeds, and multiple exploratory comparisons limit generalization claims."]
    for role,value in completed.items():
        presentation=value.get("presentation",{})
        if presentation.get("report_path"):
            relative=Path(presentation["report_path"]).relative_to(artifact)
            lines += ["",f"[{role} detailed report]({relative.as_posix()})"]
    report=artifact/"study_results.md";report.write_text("\n".join(lines)+"\n")
    result["report_path"]=str(report)
    atomic_json(artifact/"study_evaluation.json",result)
    return result


In [ ]:
assert callable(evaluate_stage) and callable(combine_stages)
print("Notebook05 evaluation definitions loaded. No features, probes, calibration, GPU work, or optimizer steps executed.")